In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [16]:

df = pd.read_csv("C:/Users/island/Desktop/train.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [17]:

print("行数：", df.shape[0])
print("列数：", df.shape[1])
print("列名：", df.columns.tolist())

行数： 891
列数： 12
列名： ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [19]:
missing_count = df.isnull().sum()
missing_rate = df.isnull().mean()

missing_table = pd.DataFrame({
    "missing_count": missing_count,
    "missing_rate": missing_rate
})

missing_table.sort_values("missing_rate", ascending=False)

,missing_count,missing_rate
Cabin,687,0.771044
Age,177,0.198653
Embarked,2,0.002245
PassengerId,0,0.000000
Name,0,0.000000
Pclass,0,0.000000
Survived,0,0.000000
Sex,0,0.000000
Parch,0,0.000000
SibSp,0,0.000000


从缺失值统计结果可以看出，部分字段存在缺失值。缺失率较高的字段需要谨慎处理，缺失较少的字段可以考虑填充。

In [20]:

df_drop = df.drop(columns=["Cabin"])

df_drop.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S


Cabin 字段缺失较多，如果直接填充可能误差较大，因此这里选择删除该列。

In [22]:

df_fill = df_drop.copy()

age_median = df_fill["Age"].median()
df_fill["Age"] = df_fill["Age"].fillna(age_median)

embarked_mode = df_fill["Embarked"].mode()[0]
df_fill["Embarked"] = df_fill["Embarked"].fillna(embarked_mode)

df_fill.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

插值法

In [23]:

df_interpolate = df_drop.copy()

df_interpolate["Age"] = df_interpolate["Age"].interpolate()

df_interpolate["Age"].isnull().sum()

np.int64(0)

In [24]:
duplicate_count = df_fill.duplicated().sum()

print("重复记录数量：", duplicate_count)

重复记录数量： 0


In [26]:

df_clean = df_fill.drop_duplicates()

print("删除重复记录后的行数：", df_clean.shape[0])
q1 = df_clean["Fare"].quantile(0.25)
q3 = df_clean["Fare"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("下限：", lower)
print("上限：", upper)

outliers = df_clean[(df_clean["Fare"] < lower) | (df_clean["Fare"] > upper)]

print("Fare 异常值数量：", outliers.shape[0])

删除重复记录后的行数： 891
下限： -26.724
上限： 65.6344
Fare 异常值数量： 116


In [28]:

df_clean["Survived"] = df_clean["Survived"].astype("category")
df_clean["Pclass"] = df_clean["Pclass"].astype("category")
df_clean["Sex"] = df_clean["Sex"].astype("category")
df_clean["Embarked"] = df_clean["Embarked"].astype("category")

df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   PassengerId  891 non-null    int64   
 1   Survived     891 non-null    category
 2   Pclass       891 non-null    category
 3   Name         891 non-null    str     
 4   Sex          891 non-null    category
 5   Age          891 non-null    float64 
 6   SibSp        891 non-null    int64   
 7   Parch        891 non-null    int64   
 8   Ticket       891 non-null    str     
 9   Fare         891 non-null    float64 
 10  Embarked     891 non-null    category
dtypes: category(4), float64(2), int64(3), str(2)
memory usage: 52.4 KB


In [29]:

df_clean.columns = df_clean.columns.str.lower().str.replace(" ", "_")

df_clean.columns

Index(['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp',
       'parch', 'ticket', 'fare', 'embarked'],
      dtype='str')

In [30]:
df_clean["sex"] = df_clean["sex"].astype(str).str.lower()

df_clean[["name", "sex", "age", "fare"]].head()

,name,sex,age,fare
0,"Braund, Mr. Owen Harris",male,22.0,7.2500
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,71.2833
2,"Heikkinen, Miss. Laina",female,26.0,7.9250
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,53.1000
4,"Allen, Mr. William Henry",male,35.0,8.0500


In [31]:
df_clean.to_csv("data/titanic_cleaned.csv", index=False, encoding="utf-8-sig")

print("清洗后的数据已保存。")

清洗后的数据已保存。
